In [35]:
import torch
from torch_geometric.datasets import KarateClub
import yaml

dataset = KarateClub()
print(f'Dataset: {dataset}:')
print('======================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

data = dataset[0]  # Get the first graph object.

# 학습 노드를 제외하고 각 클래스에서 검증 노드를 하나씩 선택
val_mask = torch.zeros_like(data.train_mask, dtype=torch.bool)

for class_id in range(dataset.num_classes):
    candidates = torch.where(
        (data.y == class_id) & (~data.train_mask)
    )[0]

    if candidates.numel() == 0:
        raise ValueError(
            f'클래스 {class_id}에 검증용으로 선택할 노드가 없습니다.'
        )

    val_mask[candidates[0]] = True

# 학습 및 검증에 사용하지 않은 나머지 노드를 테스트에 사용
test_mask = ~(data.train_mask | val_mask)

data.val_mask = val_mask
data.test_mask = test_mask

Dataset: KarateClub():
Number of graphs: 1
Number of features: 34
Number of classes: 4


In [36]:
# Load best config from wandb sweep
# path = "./wandb/sweep-dl7hw59b/config-9grjgorg.yaml" # (GCN) put best config path here 
# path = "./wandb/sweep-9ywm6966/config-7ykidmt5.yaml" # (GAT) put best config path here 
# path = "./wandb/sweep-dmk1nflb/config-637nrmi1.yaml" # (GraphSAGE) put best config path here 
path = "./wandb/sweep-4qcu101d/config-q7jpjj34.yaml" # (DNN) put best config path here 

with open(path, "r", encoding="utf-8") as f:
    raw_config = yaml.safe_load(f)

# W&B YAML stores each parameter as {value: ...}.
best_config = {
    key: entry["value"]
    for key, entry in raw_config.items()
    if isinstance(entry, dict) and "value" in entry
}

def config_kwargs(prefix):
    return {
        key[len(prefix):]: value # round(value, 3)
        for key, value in best_config.items()
        if key.startswith(prefix)
    }

In [37]:
best_config

{'models.DNN.dropout': 0.0254623346010985,
 'models.DNN.hidden_channels': 32,
 'models.DNN.num_layers': 2,
 'optimizer.Adam.lr': 0.0002846197055623514,
 'optimizer.Adam.weight_decay': 0.00011437683156224582,
 'training.gradient_clip_val': 1.9283749305083375,
 'wandb_project': 'karate-club-gcn-sweep'}

In [38]:
import torch
import torch.nn.functional as F

from utils.model import GCN, GAT, GraphSAGE, DNN

import random

def seed_everything(seed):
    """ Set random seeds for reproducibility.
    Args:
        seed (int): Random seed.
    """
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(0)
# model_kwargs = config_kwargs("models.GCN.")
# model = GCN(in_channels=dataset.num_features,
#             num_classes=dataset.num_classes,
#             **model_kwargs)
# model_kwargs = config_kwargs("models.GAT.")
# model = GAT(in_channels=dataset.num_features,
#             num_classes=dataset.num_classes,
#             **model_kwargs)
# model_kwargs = config_kwargs("models.GraphSAGE.")
# model = GraphSAGE(in_channels=dataset.num_features,
#             num_classes=dataset.num_classes,
#             **model_kwargs)
model_kwargs = config_kwargs("models.DNN.")
model = DNN(in_channels=dataset.num_features,
            num_classes=dataset.num_classes,
            **model_kwargs)

print("Model config:", model_kwargs)
print(model)

Model config: {'dropout': 0.0254623346010985, 'hidden_channels': 32, 'num_layers': 2}
DNN(
  (fcs): ModuleList(
    (0): Linear(in_features=34, out_features=32, bias=True)
    (1): Linear(in_features=32, out_features=32, bias=True)
  )
  (classifier): Linear(in_features=32, out_features=4, bias=True)
)


In [39]:
criterion = torch.nn.CrossEntropyLoss()  # Define loss criterion.
optimizer_kwargs = config_kwargs("optimizer.Adam.")
# PyYAML may parse scientific notation as a string.
optimizer_kwargs = {key: float(value) for key, value in optimizer_kwargs.items()}
optimizer = torch.optim.Adam(model.parameters(), **optimizer_kwargs)
gradient_clip_val = float(best_config["training.gradient_clip_val"])
print("Adam config:", optimizer_kwargs)
print("Gradient clip:", gradient_clip_val)

def train(data):
    model.train()
    optimizer.zero_grad()  # Clear gradients.
    out = model(data.x, data.edge_index)  # Perform a single forward pass.
    loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Compute the loss solely based on the training nodes.
    loss.backward()  # Derive gradients.
    if gradient_clip_val > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_val)
    optimizer.step()  # Update parameters based on gradients.
    return loss

loss_history = []
for epoch in range(500):
    loss = train(data)
    loss_history.append(loss.item())
    if epoch % 10 == 0:
        print(f"epoch={epoch}, loss={loss.item():.4f}")

Adam config: {'lr': 0.0002846197055623514, 'weight_decay': 0.00011437683156224582}
Gradient clip: 1.9283749305083375
epoch=0, loss=1.3856
epoch=10, loss=1.4057
epoch=20, loss=1.3879
epoch=30, loss=1.3748
epoch=40, loss=1.3838
epoch=50, loss=1.3819
epoch=60, loss=1.3695
epoch=70, loss=1.3618
epoch=80, loss=1.3537
epoch=90, loss=1.3501
epoch=100, loss=1.3401
epoch=110, loss=1.3305
epoch=120, loss=1.3417
epoch=130, loss=1.3158
epoch=140, loss=1.3050
epoch=150, loss=1.2839
epoch=160, loss=1.2861
epoch=170, loss=1.2623
epoch=180, loss=1.2335
epoch=190, loss=1.2272
epoch=200, loss=1.2043
epoch=210, loss=1.1821
epoch=220, loss=1.1525
epoch=230, loss=1.1326
epoch=240, loss=1.1010
epoch=250, loss=1.0688
epoch=260, loss=1.0641
epoch=270, loss=0.9924
epoch=280, loss=0.9619
epoch=290, loss=0.9412
epoch=300, loss=0.9045
epoch=310, loss=0.9138
epoch=320, loss=0.8387
epoch=330, loss=0.8226
epoch=340, loss=0.7994
epoch=350, loss=0.7257
epoch=360, loss=0.6652
epoch=370, loss=0.6340
epoch=380, loss=0.61

In [ ]:
# import numpy as np

# np.save("./loss_curve_vis/data/dnn_train_loss_hpt.npy", np.array(loss_history))

In [41]:
model.eval()
logits = model(data.x, data.edge_index)

test_loss = F.cross_entropy(logits[data.test_mask], data.y[data.test_mask]).item()
test_accuracy = (
        F.softmax(logits[data.test_mask],dim=1).argmax(dim=1) == data.y[data.test_mask]
    ).float().mean().item()

print("test loss: ", test_loss)
print("test accuracy: ", test_accuracy)

test loss:  1.3170850276947021
test accuracy:  0.5384615659713745


In [ ]:
# 예측 값
F.softmax(logits[data.test_mask],dim=1).argmax(dim=1)

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1])

In [ ]:
# 실제 값
data.y[data.test_mask]

tensor([1, 1, 1, 3, 3, 1, 1, 3, 1, 1, 1, 0, 0, 3, 1, 0, 1, 0, 1, 0, 0, 2, 0, 0,
        2, 0, 0, 2, 0, 0])